# 00 · Setup and your first request

**Goal:** go from a fresh clone to a working Claude API call, and understand every field that comes back.

| | |
|---|---|
| **You will learn** | How a request travels, which model to pick, how to check cost before paying, how to read a response, and what errors look like |
| **Prerequisites** | Python 3.10+ and an Anthropic API key |
| **Run time** | About 1 minute |
| **Cost** | About $0.005 when I ran it. This notebook stops itself at $0.10 |
| **Paid cells** | Sections 5, 6 and 7. Everything else here is free |

## Keeping spending under control

Three simple limits, from the widest to the narrowest:

1. **A spend limit on your account.** In the [Claude Console](https://platform.claude.com/settings/billing), open **Billing**, then under **Spend limits** click **Set limit**. Anthropic enforces this one: once you reach it, requests fail until the limit resets. **Do this first.**
2. **A budget in each notebook.** The `track()` function in section 2 adds up what the notebook has spent and stops it at `BUDGET_USD`.
3. **`max_tokens` on every call.** This limits how much a single answer can cost.

## 1. How a request travels

Think of a restaurant. The customer (your app) never walks into the kitchen. They tell the waiter (your server), the waiter passes the order to the kitchen (the Claude API), and the food comes back the same way.

```
 Your app        Your server             Claude API             Model
 (browser) ────▶ holds the API key ────▶ checks the key ──────▶ generates text
           ◀──── returns the text ◀───── returns the response ◀
```

**Why the server in the middle?** Your API key is a secret that spends your money. Any key shipped inside a web page or mobile app can be extracted by anyone who downloads it. So keys stay on servers, never in client code. In these notebooks, your laptop plays the part of the server, and the key stays in a gitignored `.env` file.

**What happens inside the model:**

| Stage | What it does |
|---|---|
| **Tokenization** | Splits your text into tokens: whole words, word pieces, or symbols |
| **Embedding** | Turns each token into a list of numbers that captures its possible meanings |
| **Contextualization** | Adjusts those numbers using the surrounding tokens, so "bank" settles on river or money |
| **Generation** | Predicts the next token, adds it, and repeats until something tells it to stop |

Generation stops for one of a few reasons, and the response tells you which. Section 6 uses that.

## 2. Setup

Do this once, from the repo root, in a terminal:

```bash
uv venv --python 3.12 .venv
source .venv/bin/activate
uv pip install -r requirements.txt
cp .env.example .env
```

Open `.env` and paste your key into `ANTHROPIC_API_KEY`. Then start Jupyter with `jupyter lab`, open this notebook, and run the cells below.

In [1]:
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


**Tracking what we spend.** Every API response reports the tokens it used, so working out the cost is just multiplication. We wrap each paid call in `track()`, which prints the cost, adds it to a running total, and stops the notebook once the total passes `BUDGET_USD`.

The check happens after each call, so a notebook can go over its budget by at most one call.

In [2]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.10     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response

## 3. Which model? (free)

Claude comes in tiers. Higher tiers are more capable and cost more.

| Model | Model ID | Input / Output per 1M tokens | Good for |
|---|---|---|---|
| Haiku 5.5 | `claude-haiku-5-5` | $0.10 / $0.50 | High-volume, simple, fast work |
| **Sonnet 5.5** | `claude-sonnet-5-5` | $2 / $10 | Everyday work. **Used across this repo** |
| Opus 5.5 | `claude-opus-5-5` | $4 / $20 | The hardest reasoning and long agent runs |

Prices were last checked on 5 November 2026. See the [pricing page](https://platform.claude.com/docs/en/about-claude/pricing) for current ones.

Output tokens cost five times as much as input on Sonnet, so **long answers are what get expensive**, not long questions.

You don't have to rely on a table like this one. The **Models API** is free and tells you what each model supports right now.

In [3]:
model_info = client.models.retrieve(MODEL)

print(f"{model_info.display_name} ({model_info.id})")
print(f"  Max input tokens : {model_info.max_input_tokens:,}")
print(f"  Max output tokens: {model_info.max_tokens:,}")

print("\nCapabilities:")
for name, info in model_info.capabilities.model_dump().items():
    if isinstance(info, dict) and "supported" in info:
        print(f"  {'yes' if info['supported'] else 'no ':<4}{name}")

assert model_info.id == MODEL

Claude Sonnet 5.5 (claude-sonnet-5-5)
  Max input tokens : 1,000,000
  Max output tokens: 128,000

Capabilities:
  yes batch
  yes citations
  yes code_execution
  yes context_management
  yes effort
  yes image_input
  yes pdf_input
  yes server_tools
  yes structured_outputs
  yes thinking


**What to notice**

- `max_input_tokens` is how much the model can read in one request. `max_tokens` is the most it can write.
- Checking capabilities in code is more reliable than trusting a course or a blog post, including this notebook.

## 4. Check the cost before you pay (free)

The `count_tokens` endpoint is free and reports how many input tokens a request would use. That lets you work out the most a call can cost before you make it:

**most it can cost = (input tokens × input price) + (`max_tokens` × output price)**

A token is not a word. Run the cell below and see how text splits.

In [4]:
samples = [
    "Hello",
    "Hello, world!",
    "antidisestablishmentarianism",
    "INC-2023-Q4-011",
    "The quick brown fox jumps over the lazy dog.",
]

print("Tokens  Text    (each count includes a few tokens of message wrapping)")
counts = {}
for text in samples:
    result = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": text}])
    counts[text] = result.input_tokens
    print(f"{result.input_tokens:>6}  {text}")

assert counts["The quick brown fox jumps over the lazy dog."] > counts["Hello"]

Tokens  Text    (each count includes a few tokens of message wrapping)
    11  Hello
    14  Hello, world!
    19  antidisestablishmentarianism
    19  INC-2023-Q4-011
    26  The quick brown fox jumps over the lazy dog.


In [5]:
question = [{"role": "user", "content": "What is quantum computing? Answer in 2 sentences."}]
MAX_TOKENS = 1024

input_tokens = client.messages.count_tokens(model=MODEL, messages=question).input_tokens
most_it_can_cost = (input_tokens * INPUT_PRICE + MAX_TOKENS * OUTPUT_PRICE) / 1_000_000

print(f"Input tokens     : {input_tokens}")
print(f"max_tokens       : {MAX_TOKENS}")
print(f"Most it can cost : ${most_it_can_cost:.5f}")

Input tokens     : 27
max_tokens       : 1024
Most it can cost : $0.01029


**What to notice**

- A long or unusual string like `INC-2023-Q4-011` can take several tokens, even though it reads as one "word".
- `max_tokens` drives that estimate, because output is the expensive side. The real cost is normally much lower, as the next section shows.

## 5. Your first request (paid)

Three arguments are required:

| Argument | What it is |
|---|---|
| `model` | Which Claude model to use |
| `max_tokens` | A **safety limit** on the response length. It is not a target: Claude writes what it needs and stops, up to this limit |
| `messages` | The conversation so far, as a list of `{"role", "content"}` dictionaries |

We also pass `output_config={"effort": "low"}`. Effort controls how hard Claude works on an answer. Section 7 explains why every notebook here sets it.

In [6]:
message = track(client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    output_config={"effort": "low"},
    messages=question,
))

print(f"Most it could have cost was ${most_it_can_cost:.5f}\n")
message

[input=27 output=95 cost=$0.00100 | spent $0.0010 of $0.10]
Most it could have cost was $0.01029



Message(id='msg_011CgjeyNqHVkRUBRAb8RXuT', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum bits (qubits), which can exist in multiple states at once through superposition and can be linked through entanglement. This lets quantum computers tackle certain problems, like factoring large numbers or simulating molecules, far faster than classical computers can.', type='text')], diagnostics=None, model='claude-sonnet-5-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=27, output_tokens=95, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

That is the whole response object. The fields you will use most:

| Field | Meaning |
|---|---|
| `content` | A **list of blocks**. The answer is in here |
| `stop_reason` | Why generation ended |
| `usage` | Tokens used, which is what you are billed for |
| `model` | The model that answered |
| `role` | Always `assistant` |
| `id` | A unique ID, useful when debugging |

### Reading the answer safely

`content` is a list because a response can hold several kinds of block: `thinking`, `text`, `tool_use`, and more. Sonnet 5.5 can think before it answers, so **the first block may not be text**, and `message.content[0].text` can crash.

Look for the block whose type is `text` instead. We use this small function in every notebook.

In [7]:
print("Block types:", [block.type for block in message.content])


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


answer = text_from_message(message)
print("\nAnswer:\n" + answer)

assert message.role == "assistant"
assert message.stop_reason == "end_turn"
assert answer

Block types: ['text']

Answer:
Quantum computing is a type of computing that uses quantum bits (qubits), which can exist in multiple states at once through superposition and can be linked through entanglement. This lets quantum computers tackle certain problems, like factoring large numbers or simulating molecules, far faster than classical computers can.


## 6. `max_tokens` and stop reasons (paid)

Now we set `max_tokens` too low on purpose. Claude doesn't plan around the limit. It simply gets cut off, and `stop_reason` tells you so.

In [8]:
cut_off = track(client.messages.create(
    model=MODEL,
    max_tokens=60,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Name 30 countries, one per line, with their capitals."}],
))

print("stop_reason:", cut_off.stop_reason)
print("\nWhat came back:\n" + (text_from_message(cut_off) or "(no text: the limit was used up before any answer)"))

assert cut_off.stop_reason == "max_tokens"

[input=26 output=60 cost=$0.00065 | spent $0.0017 of $0.10]
stop_reason: max_tokens

What came back:
1. France – Paris
2. Japan – Tokyo
3. Brazil – Brasília
4. Canada – Ottawa
5. Australia – Canberra
6. Egyp


**What to notice**

- `output` matches the limit, and the answer stops mid-list.
- **Thinking counts toward `max_tokens`.** If you set it very low, thinking can use all of it and you get no text at all.
- Always check `stop_reason` in real code. A cut-off answer can look like a complete one.

| `stop_reason` | Meaning | What to do |
|---|---|---|
| `end_turn` | Claude finished naturally | Use the answer |
| `max_tokens` | Hit your limit | Raise `max_tokens`, or ask for less |
| `stop_sequence` | Hit a stop string you set | Expected. See notebook 01 |
| `tool_use` | Claude wants to run a tool | Run it and send back the result. See notebook 05 |
| `pause_turn` | A long server-side tool turn paused | Send it back to continue. See notebook 06 |
| `refusal` | Claude declined the request | Read the response, and don't retry it unchanged |

## 7. Why every notebook sets `effort` (paid)

Effort is the main dial for how much thinking Claude does, and thinking is billed as output tokens. It runs from `low` to `max`, and **Sonnet 5.5 defaults to `high`**. So leaving it out can cost more than you expect on a harder task.

Here is a small puzzle at two effort levels:

In [10]:
puzzle = [{"role": "user", "content": (
    "A farmer has 17 sheep. All but 9 run away, then he buys twice as many as remain. "
    "How many sheep does he have? Answer with just the number."
)}]

for effort in ["low", "high"]:
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=1500,
        output_config={"effort": effort},
        messages=puzzle,
    ))
    print(f"effort={effort}: answer = {text_from_message(response)}\n")
    assert response.stop_reason == "end_turn"

[input=57 output=50 cost=$0.00061 | spent $0.0036 of $0.10]
effort=low: answer = "All but 9 run away" means 9 remain. He buys twice as many as remain: 18. Total = 9 + 18 = 27.

**27**

[input=57 output=51 cost=$0.00062 | spent $0.0042 of $0.10]
effort=high: answer = "All but 9 run away" means 9 remain. He then buys twice as many as remain: 18. Total = 9 + 18 = 27.

**27**



**What to notice**

- **The two runs cost almost the same.** When I ran this, `low` used 50 output tokens and `high` used 51. Thinking is adaptive, so Claude skips it when a question is easy, whatever the effort setting. Your numbers may differ.
- **Effort isn't a fixed price.** It matters on harder problems, where higher effort can mean much more thinking, and you pay for that thinking as output even though you never see it. Notebook 09 tries this on a problem hard enough to show the gap.
- **Claude showed its working, even though we asked for just the number.** With no thinking step, the reasoning has to go somewhere. Notebook 09 covers this too.
- We set `effort` explicitly in every notebook so the cost never depends on a default.

## 8. Errors, and the budget stop (free)

None of the cells in this section are billed: each one fails before any tokens are generated.

| Error | Usual cause | What to do |
|---|---|---|
| `AuthenticationError` (401) | Wrong or missing API key | Check `.env` |
| `BadRequestError` (400) | The request is wrong, or you reached your own Console spend limit | Read the message |
| `NotFoundError` (404) | Wrong model name | Check `client.models.list()` |
| `RateLimitError` (429) | Too many requests, or your tier's monthly cap | Read the message. The SDK retries twice for you |
| `InternalServerError` (5xx) | A problem on Anthropic's side | Wait. The SDK retries twice for you |

**A wrong model name:**

In [11]:
import anthropic

try:
    client.models.retrieve("claude-does-not-exist")
    raise AssertionError("Expected a NotFoundError")
except anthropic.NotFoundError as error:
    print(f"NotFoundError, status {error.status_code}")

NotFoundError, status 404


**`temperature` no longer exists.** Older tutorials, and the course videos, use `temperature` to control randomness. The current API version rejects `temperature`, `top_p` and `top_k` on every model, so SDK 2.x removed those arguments. The call now fails in Python, before anything is sent.

In [12]:
try:
    client.messages.create(
        model=MODEL,
        max_tokens=50,
        temperature=0.5,
        messages=[{"role": "user", "content": "Hi"}],
    )
    raise AssertionError("Expected a TypeError")
except TypeError as error:
    print(f"TypeError: {error}")

TypeError: Messages.create() got an unexpected keyword argument 'temperature'


**Seeing the budget stop.** No API call here. We pretend the whole budget is already spent, then pass `track()` the response we already have from section 5.

In [13]:
actually_spent = spent
spent = BUDGET_USD

try:
    track(message)
    raise AssertionError("Expected track() to stop the notebook")
except RuntimeError as error:
    print(f"Stopped: {error}")

spent = actually_spent

[input=27 output=95 cost=$0.00100 | spent $0.1010 of $0.10]
Stopped: Budget reached: spent $0.1010. Raise BUDGET_USD if you mean to spend more.


## 9. Final spend

In [14]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0042 of its $0.10 budget.


## Takeaways

1. **Keep the API key on a server**, never in client code.
2. **`model`, `max_tokens` and `messages` are required.** `max_tokens` is a limit, not a target.
3. **`content` is a list of blocks.** Find the `text` block instead of assuming `content[0]`.
4. **Check `stop_reason`**, because a cut-off answer can look complete.
5. **`count_tokens` and the Models API are free.** Use them before spending.
6. **Output costs more than input, and thinking counts as output.** Set `effort` on purpose.

## Your turn

Each exercise costs about a cent or less, and the last one is free.

- In section 5, change the question and compare the cost of a short and a long answer.
- In section 6, try `max_tokens=20`. Do you still get any text?
- In section 7, add `"medium"` to the list of effort levels. Where does it land?
- Remove `max_tokens` from a call and read the error. Which layer caught it, the SDK or the API?

**Next:** [01 · Conversations and control](01_conversations_and_control.ipynb): multi-turn chat, system prompts, streaming and stop sequences.